In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, pipeline, logging
from trl import SFTTrainer
from peft import LoraConfig, PeftModel, TaskType
import torch
import json
from datasets import load_dataset

In [ ]:
import pandas as pd

df = pd.read_csv("train2.csv")
print(df.head())

In [ ]:
df = pd.read_csv("train2.csv")

# keep only needed columns
df = df[["question1", "question2", "is_duplicate"]]

# convert to JSONL
df.to_json("train2.jsonl", orient="records", lines=True)

In [ ]:
import json
from datasets import Dataset

# Load JSONL
with open("train2.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

# Convert into instruction format
for d in data:
    d["text"] = f"""### Instruction:
Determine if the two questions are semantically duplicate.

### Input:
Question 1: {d['question1']}
Question 2: {d['question2']}

### Response:
{"Yes" if d["is_duplicate"] == 1 else "No"}"""

# Create HF dataset
dataset = Dataset.from_list(data)

# Keep only text column (important)
dataset = dataset.remove_columns(
    ["question1", "question2", "is_duplicate"]
)

print(dataset)

In [ ]:
dataset['text']

### Load model and tokenizer

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# base_model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", torch_dtype=torch.float16)

In [ ]:
base_model

### Baseline Generation

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer
)

q1 = "What is AI?"
q2 = "What is artificial intelligence?"

prompt = f"""
### Instruction:
Determine if the two questions are semantically duplicate.

### Input:
Question 1: {q1}
Question 2: {q2}

### Response: {"Duplicate" if d["is_duplicate"] == 1 else "Not duplicate"}
"""

result = pipe(
    prompt,
    max_new_tokens=5,
    do_sample=False
)

print(result[0]["generated_text"])

### Configure LORA

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Adjust based on model architecture
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

### Training Configuration

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "True"

In [ ]:
training_args = TrainingArguments(
    output_dir="./tinyllama-duplicate-detector",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=1,

    learning_rate=2e-4,

    logging_steps=20,

    save_strategy="epoch",
    report_to="none",

    fp16=True,

    lr_scheduler_type="cosine",

    warmup_ratio=0.03,

    save_total_limit=2
)

In [ ]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=64)


dataset = dataset.map(tokenize, batched=True)

In [ ]:
# Trainer
trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args,
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained("D:\Desktop\Quora_Duplicate_Project")
tokenizer.save_pretrained("D:\Desktop\Quora_Duplicate_Project")

In [ ]:
# Robust test: use the same prompt format you used during training
import re

q1 = "How can I lose weight naturally?"
q2 = "What are natural methods for weight loss?"

prompt = f"""### Instruction:
Determine if the two questions are semantically duplicate.

### Input:
Question 1: {q1}
Question 2: {q2}

### Response:
"""

out = pipe(
    prompt,
    max_new_tokens=3,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

raw_answer = out[0]["generated_text"].strip()
first_token = re.sub(r"[^a-z]", "", raw_answer.split()[0].lower()) if raw_answer else ""

if first_token in ["yes", "duplicate"]:
    pred = "Duplicated"
elif first_token in ["no", "not"]:
    pred = "Not duplicated"
else:
    pred = f"Unclear output: {raw_answer}"

print("Final prediction:", pred)

[transformers] Both `max_new_tokens` (=3) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Final prediction: Duplicated
